# Bài 09 · Gradient biến mất & khởi tạo: đo tín hiệu đi qua một mạng MNIST 20 lớp

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/09-vanishing-gradient.ipynb)

Notebook đi kèm [Bài 09 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vanishing-gradient/) (demo tương tác nằm ở trang bài học).
Vì sao mạng sâu từng rất khó huấn luyện? Ta dựng lại đúng mạng của demo, một MLP 784 → 20 lớp × 64 → 10 với cùng
từng trọng số, cho 50 ảnh MNIST thật đi qua, rồi đo xem tín hiệu và gradient còn lại bao nhiêu sau mỗi lớp.

**Bạn sẽ làm:**
1. Nhân một thừa số 30 lần, rồi kiểm công thức $\prod_l w_l\, f'(z_l)$ trên một chuỗi 30 nơ-ron.
2. Tái tạo bit-exact trọng số của demo (bộ sinh mulberry32 + Box–Muller) và đo gradient ở từng lớp của mạng 20 lớp.
3. Tự suy ra thừa số mỗi lớp $\sigma\sqrt{n}\cdot\operatorname{rms}(f')$, rồi tìm cỡ khởi tạo giữ cho nó bằng 1: Xavier, He.
4. Huấn luyện thật 1 epoch: thấy lớp đầu đứng yên, He thắng Xavier với ReLU, và cắt gradient chặn được bùng nổ.
5. Đọc kết quả quét đầy đủ của demo: 3 hàm kích hoạt × 4 cỡ khởi tạo × 4 độ sâu × có/không cắt gradient × 5 giá trị η.

**Cần biết trước:** [Bài 02 · Hàm kích hoạt](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-kich-hoat/)
(sigmoid, tanh, ReLU, $\sigma'(0) = 0{,}25$, bão hòa) và
[Bài 06 · Lan truyền ngược](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/) (lượt ngược dạng ma
trận, dùng lại ở mục 4). Tốc độ học $\eta$ và cách chọn nó trên tập kiểm định là của
[Bài 07](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo. Riêng kết quả quét đầy đủ
(480 lần huấn luyện) được nạp từ đúng tệp mà demo dùng, vì chạy lại mất vài phút; notebook chạy lại vài cấu hình của
nó và kiểm tra rằng ra cùng con số.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi. Bài này gặp những
con số rất nhỏ và rất lớn, nên `num()` in chúng như demo: số thập phân thường từ 0,001 đến 10.000, ngoài khoảng đó
là dạng $8{,}5 \cdot 10^{-13}$.

In [ ]:
import base64
import gzip
import hashlib
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = (
        f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    )  # không in “−0”
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 8,5·10⁻¹³."""
    if x == 0:
        return "0"
    m, e = f"{x:.{digits}e}".split("e")
    return f"{vn(float(m), digits)}·10{str(int(e)).translate(SUP)}"


def num(x):
    """Như demo: số thường từ 0,001 đến 10.000 (0,0042 · 0,231 · 17,45 · 1.074), ngoài khoảng đó dạng 10ᵏ."""
    a = abs(x)
    if 1e-3 <= a < 1e4:
        return vn(x, 4 if a < 0.01 else 3 if a < 1 else 2 if a < 100 else 0)
    return sci(x)


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(vn(13007, 0), vn(0.1, 1), pct(0.9239), num(0.2316), num(17.449), num(8.53e-13), num(2.06e14))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28. Ô dưới tải
4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu. Giống demo và Bài 07, ta chia ba phần:
55.000 ảnh đầu để huấn luyện, 5.000 ảnh cuối của tệp huấn luyện để *chọn* $\eta$ (kiểm định), và 10.000 ảnh kiểm tra
chỉ để báo kết quả cuối.

Ô cũng nạp `mnist-deep.json`, tệp kết quả mà demo dùng (`fetch_json`, có kiểm tra MD5). Nó chứa 50 ảnh để đo tín hiệu
và kết quả của 480 lần huấn luyện mà ta sẽ đọc ở mục 8.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels.astype(np.int64)


# Kết quả của một lần quét dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
train_img, y_all = load("train")
test_img, yte = load("t10k")
DEEP = fetch_json("mnist-deep.json", "b60ee8e1b1e12d7d6f76037f8b662db2")  # kết quả quét của demo
X_all = train_img / 255.0
Xtr, ytr = X_all[:55000], y_all[:55000]  # huấn luyện
Xval, yval = X_all[55000:], y_all[55000:]  # kiểm định: chọn η ở đây
Xte = test_img / 255.0  # kiểm tra: chỉ để báo cáo
print(f"huấn luyện {vn(len(ytr), 0)} · kiểm định {vn(len(yval), 0)} · kiểm tra {vn(len(yte), 0)} ảnh")
assert (len(ytr), len(yval), len(yte)) == (DEEP["nTrain"], DEEP["nVal"], DEEP["nTest"])

Demo đo tín hiệu trên một lô nhỏ cố định: **5 ảnh kiểm tra đầu tiên của mỗi chữ số**, tổng cộng 50 ảnh. Ô dưới chọn lại
đúng các ảnh đó và kiểm tra rằng chúng trùng từng byte với ảnh trong tệp của demo.

In [ ]:
pick = np.concatenate([np.flatnonzero(yte == d)[:5] for d in range(10)])
xb, yb = Xte[pick], yte[pick]  # lô 50 ảnh, x cỡ 50 × 784
demo_imgs = np.frombuffer(base64.b64decode(DEEP["images"]), np.uint8).reshape(-1, 784)
assert list(pick) == DEEP["mnist"] and (demo_imgs == test_img[pick]).all(), "khác với lô 50 ảnh của demo"
print(f"lô {xb.shape[0]} ảnh: chỉ số {pick[:5].tolist()}… trùng với demo")

fig, axes = plt.subplots(5, 10, figsize=(8, 4.2))
for k, ax in enumerate(axes.T.flat):  # mỗi cột một chữ số
    ax.imshow(xb[k].reshape(28, 28), cmap="gray_r")
    ax.axis("off")
for d in range(10):
    axes[0, d].set_title(str(d), fontsize=11)
plt.show()

Đó là 50 ảnh của lô, mỗi cột một chữ số. Mọi phép đo tín hiệu ở mục 4–5 và 7 đều chạy trên đúng lô này, như demo.

## 3 · Nhân một thừa số ba mươi lần

Hãy tưởng tượng trò chơi truyền tin qua 30 người: mỗi người nói lại cho người kế tiếp nhỏ hơn mình nghe một chút.
Đến người cuối, tin còn lại bao nhiêu? Lan truyền ngược (Bài 06) chạy đúng như vậy: gradient đi từ lớp cuối về lớp
đầu, và ở mỗi lớp bị nhân với một thừa số. Bài 02 đã tính $0{,}25^L$, cận trên của tích các $\sigma'$. Ô dưới cho
thấy thừa số không cần nhỏ cỡ 0,25 mới gây chuyện: chỉ cần lệch khỏi 1 một chút là đủ.

In [ ]:
for k in [0.25, 0.5, 0.9, 0.99, 1.0, 1.01, 1.1, 2.0]:
    print(f"{vn(k, 2):>4} nhân 30 lần = {num(k**30)}")
assert f"{0.25**30:.2e}" == "8.67e-19" and f"{0.9**30:.3f}" == "0.042" and f"{1.1**30:.2f}" == "17.45"

Thừa số 0,9 nghe vô hại, nhưng qua 30 lớp chỉ còn 4% tín hiệu; 1,1 thì phình gấp 17 lần. Chỉ đúng 1 mới giữ nguyên.

Thừa số ấy là gì? Lấy mạng đơn giản nhất: một chuỗi 30 nơ-ron, mỗi lớp một nơ-ron, $h_l = f(z_l)$ với
$z_l = w_l h_{l-1}$ (không có độ lệch). Theo quy tắc chuỗi, $\partial h_l / \partial h_{l-1} = f'(z_l)\, w_l$, nên

$$\frac{\partial h_{30}}{\partial h_1} = \prod_{l=2}^{30} w_l\, f'(z_l)$$

Thừa số mỗi lớp là $w_l f'(z_l)$, không chỉ $f'(z_l)$. Ô dưới định nghĩa ba hàm kích hoạt của bài (cùng công thức với
demo), tính tích đó cho một chuỗi tanh với $w_l \sim \mathcal N(0, 1)$, rồi kiểm nó bằng cách nhích thử $h_1$ (sai
phân hữu hạn, như Bài 06).

In [ ]:
def act(name, z):
    """f(z) và f′(z) của sigmoid, tanh, ReLU, viết đúng như demo và reel/mnist_mlp/deep.py."""
    if name == "sigmoid":
        with np.errstate(over="ignore"):  # z rất âm: e^−z tràn thành vô cùng, và 1/(1 + ∞) = 0 vẫn đúng
            h = 1 / (1 + np.exp(-z))
        return h, h * (1 - h)
    if name == "tanh":
        h = np.tanh(z)
        return h, 1 - h * h
    if name == "relu":  # quy ước f′(0) = 0
        return np.maximum(0, z), (z > 0).astype(float)
    raise ValueError(name)


def chain(f, w, h1):
    """Chuỗi một nơ-ron mỗi lớp: h_l = f(w_l·h_{l−1}). Trả về h_1…h_30 và 29 thừa số w_l·f′(z_l)."""
    hs, factors = [h1], []
    for wl in w:
        h, d = act(f, wl * hs[-1])
        hs.append(h)
        factors.append(wl * d)
    return np.array(hs), np.array(factors)


eps_chain = np.random.default_rng(0).normal(0, 1, 29)  # w_2 … w_30
_, factors = chain("tanh", eps_chain, 0.5)
nudge = 1e-6
fd = (chain("tanh", eps_chain, 0.5 + nudge)[0][-1] - chain("tanh", eps_chain, 0.5 - nudge)[0][-1]) / (2 * nudge)
print(f"tanh, w ~ N(0, 1): tích 29 thừa số w·f′ = {sci(np.prod(factors), 4)};  nhích thử h₁: {sci(fd, 4)}")
geo = np.exp(np.log(np.abs(factors)).mean())  # trung bình nhân: thừa số "điển hình" mỗi lớp
print(f"|thừa số| từ {vn(np.abs(factors).min(), 4)} đến {vn(np.abs(factors).max(), 2)}; trung bình nhân {vn(geo, 3)}")
assert abs(fd / np.prod(factors) - 1) < 1e-6

Hai cách cho cùng một số tới chữ số thứ năm: công thức tích là đúng. Có thừa số lớn hơn 1, có thừa số gần 0; cái
quyết định là **trung bình nhân** của chúng, vì $|\prod_l a_l| = (\text{trung bình nhân})^{29}$. Ở đây nó nhỏ hơn 1
nhiều, nên tích co về cỡ $10^{-10}$.

Hình dưới vẽ $|\partial h_k / \partial h_1|$ theo $k$ cho ba chuỗi, trục dọc thang log: tanh (tròn xanh), sigmoid với
cùng các $w$ (vuông cam), và sigmoid với $w$ lớn gấp 4 (tam giác xanh lá). Đường gạch xám là $0{,}25^{k-1}$, cận của
riêng tích các $\sigma'$ ở Bài 02.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
k = np.arange(1, 31)
for f, scale, color, marker, label in [
    ("tanh", 1, BLUE, "o", "tanh, w ~ N(0, 1)"),
    ("sigmoid", 1, ORANGE, "s", "sigmoid, cùng các w"),
    ("sigmoid", 4, GREEN, "^", "sigmoid, w lớn gấp 4"),
]:
    run_prod = np.abs(np.r_[1, np.cumprod(chain(f, scale * eps_chain, 0.5)[1])])  # |∂h_k/∂h_1|, k = 1…30
    ax.plot(k, run_prod, color=color, marker=marker, ms=4, lw=1.5, label=label)
    print(f"{label:>20}: |∂h₃₀/∂h₁| = {sci(run_prod[-1])}")
ax.plot(k, 0.25 ** (k - 1), color=GREY, ls="--", lw=1.5, label="0,25^(k−1): riêng tích σ′")
print(f"{'0,25^29':>20}: {sci(0.25**29)}")
ax.set_yscale("log")
ax.set(xlabel="lớp k", ylabel="|∂h_k / ∂h₁|", title="Chuỗi 30 nơ-ron: tích các thừa số w·f′")
ax.legend(frameon=False, fontsize=9, loc="lower left")
ax.grid(alpha=0.3)
vn_axes(fig)
plt.show()

Tanh với $w \sim \mathcal N(0, 1)$ co dần đều, dù $\tanh'$ có thể bằng 1. Sigmoid với cùng các $w$ co nhanh hơn
cả $0{,}25^{k-1}$, vì nhiều $|w_l| < 1$. Còn sigmoid với $w$ lớn gấp 4 thì vượt *trên* đường xám: trọng số lớn bù lại
giới hạn 0,25 của $\sigma'$. Thừa số thật là $w \cdot f'$; cả hai nửa đều quan trọng.

## 4 · Mạng của demo: 784 → 20 lớp × 64 → 10

Mạng thật có một ma trận ở mỗi lớp thay vì một số $w_l$. Demo dùng một MLP “trơn” (không chuẩn hóa, không kết nối tắt):
$h_l = f(h_{l-1}W_l + b_l)$, $L$ lớp ẩn rộng 64, rồi 10 logit và softmax. Để kết quả của ta trùng **từng bit** với
demo, ta sinh trọng số đúng như trình duyệt sinh:
- bộ sinh số ngẫu nhiên **mulberry32** với seed 1: mỗi lần gọi băm một số nguyên 32 bit thành một số đều trong $[0, 1)$;
- biến đổi **Box–Muller**: mỗi cặp $(u_1, u_2)$ thành một số $\varepsilon = \sqrt{-2\ln(1 - u_1)}\,\cos(2\pi u_2)$ có phân
  phối chuẩn $\mathcal N(0, 1)$;
- các $\varepsilon$ lấp lần lượt $\varepsilon_{\text{ra}}$ (64 × 10), $\varepsilon_1$ (784 × 64), rồi
  $\varepsilon_2 \dots \varepsilon_{30}$ (64 × 64), theo hàng.

Rồi $W_l = \sigma_l\, \varepsilon_l$ và $b_l = 0$. Đổi cách khởi tạo chỉ đổi $\sigma_l$, nên mọi cấu hình dùng **cùng**
các $\varepsilon$; mạng 10 lớp là 10 lớp đầu của mạng 30 lớp.

In [ ]:
def mulberry32(seed, n):
    """n số nguyên 32 bit đầu tiên của mulberry32(seed), bộ sinh mà demo (JavaScript) dùng, viết dạng vectơ.
    Lần gọi thứ k băm seed + k·0x6D2B79F5 (mod 2³²). Mọi phép tính là số nguyên 32 bit, nên trùng từng bit với JS."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)  # phép nhân uint32 tự quay vòng mod 2³², như Math.imul
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return t ^ (t >> 14)


def draw_eps(seed=1, width=64, max_depth=30):
    """ε ~ N(0, 1) cho mọi lớp, theo đúng thứ tự của demo: ε_ra (width × 10), ε₁ (784 × width), ε₂…ε_max (width × width)."""
    shapes = [(width, 10), (784, width)] + [(width, width)] * (max_depth - 1)
    sizes = [r * c for r, c in shapes]
    u = mulberry32(seed, 2 * sum(sizes)) / 2**32  # số đều trong [0, 1)
    z = np.sqrt(-2 * np.log(1 - u[0::2])) * np.cos(2 * np.pi * u[1::2])  # Box–Muller: một số N(0, 1) mỗi cặp
    eps = [p.reshape(s) for p, s in zip(np.split(z, np.cumsum(sizes)[:-1]), shapes)]
    return eps[1:], eps[0]  # (ε₁ … ε_max), ε_ra


EPS = draw_eps(DEEP["seed"], DEEP["width"], DEEP["maxDepth"])
n_eps = sum(e.size for e in EPS[0]) + EPS[1].size
print(f"{vn(n_eps, 0)} số N(0, 1); bốn số nguyên đầu của mulberry32(1): {mulberry32(1, 4).tolist()}")
print(
    f"ε_ra[0, 0] = {vn(EPS[1][0, 0], 16)};  trung bình {vn(np.r_[EPS[1].ravel(), *[e.ravel() for e in EPS[0]]].mean(), 4)}"
)
# Giá trị in ra khi chạy chính đoạn JavaScript của demo (DeepSignalDemo.tsx) bằng node
assert mulberry32(1, 4).tolist() == [2693262067, 11749833, 2265367787, 4213581821], "khác với bộ sinh của demo"
assert abs(EPS[1][0, 0] - 1.4043387784152637) < 1e-12 and n_eps == 169600

Bốn số nguyên đầu tiên và số chuẩn đầu tiên trùng với đoạn JavaScript của demo. Giờ ta định nghĩa bốn cách chọn
$\sigma$ như demo (mục 5 giải thích từng cách), lượt xuôi, và lượt ngược của Bài 06. Lượt ngược ghi lại
$\lVert\partial\mathcal L/\partial h_l\rVert$ ở mỗi lớp ẩn: gradient còn lại bao nhiêu khi đến lớp đó.

In [ ]:
def sigma_of(init, n_in, n_out):
    """Độ lệch chuẩn của W, như demo: nhỏ 0,01 · Xavier √(2/(n_in + n_out)) · He √(2/n_in) · lớn 1.
    init cũng có thể là một số g: σ = g/√n_in (mục 5)."""
    if not isinstance(init, str):
        return init / np.sqrt(n_in)
    return {"small": 0.01, "xavier": np.sqrt(2 / (n_in + n_out)), "he": np.sqrt(2 / n_in), "large": 1.0}[init]


def init_net(init, depth, eps=EPS):
    eps_h, eps_out = eps
    width = eps_out.shape[0]
    sizes = [784] + [width] * depth
    Ws = [sigma_of(init, sizes[l], width) * eps_h[l] for l in range(depth)] + [sigma_of(init, width, 10) * eps_out]
    return Ws, [np.zeros(W.shape[1]) for W in Ws]


def forward(f, Ws, bs, x):
    """hs: x rồi h₁ … h_L; ds: f′(z) của từng lớp ẩn; trả về thêm 10 logit."""
    hs, ds = [x], []
    for W, b in zip(Ws[:-1], bs[:-1]):
        h, d = act(f, hs[-1] @ W + b)
        hs.append(h)
        ds.append(d)
    return hs, ds, hs[-1] @ Ws[-1] + bs[-1]


def ce_loss(z, y):
    """Cross-entropy trung bình, tính bằng log-sum-exp nên logit rất lớn vẫn hữu hạn."""
    zmax = z.max(1)
    return float(np.mean(zmax + np.log(np.exp(z - zmax[:, None]).sum(1)) - z[np.arange(len(y)), y]))


def backward(Ws, hs, ds, z, y):
    """Lan truyền ngược (Bài 06) cho softmax + cross-entropy: gradient của mọi W, b, và ‖∂L/∂h_l‖ của từng lớp ẩn."""
    e = np.exp(z - z.max(1, keepdims=True))
    dz = e / e.sum(1, keepdims=True)
    dz[np.arange(len(y)), y] -= 1  # ∂L/∂z = p − onehot(y), chia cho cỡ lô
    dz /= len(y)
    gW, gb, gh = [hs[-1].T @ dz], [dz.sum(0)], []
    dh = dz @ Ws[-1].T
    for l in reversed(range(len(ds))):
        gh.insert(0, np.linalg.norm(dh))
        da = dh * ds[l]  # đi qua hàm kích hoạt: nhân với f′(z)
        gW.insert(0, hs[l].T @ da)
        gb.insert(0, da.sum(0))
        dh = da @ Ws[l].T  # đi qua trọng số: nhân với Wᵀ
    return gW, gb, np.array(gh)

Ô dưới gói mọi phép đo của demo vào một hàm `signal()`, chạy trên lô 50 ảnh với mạng *chưa huấn luyện*:
- **lượt xuôi:** 50 ảnh còn khác nhau bao nhiêu ở lớp $l$ (độ lệch chuẩn giữa các ảnh của từng nơ-ron, lấy RMS qua
  64 nơ-ron), và phần nơ-ron bão hòa (sigmoid: $h < 0{,}03$ hoặc $h > 0{,}97$; tanh: $|h| > 0{,}97$) hoặc chết (ReLU:
  bằng 0 với cả 50 ảnh);
- **lượt ngược:** $\lVert\partial\mathcal L/\partial h_l\rVert$ ở từng lớp, chuẩn của toàn bộ gradient, và **thừa số mỗi
  lớp**: trung bình nhân $(\lVert\partial\mathcal L/\partial h_1\rVert / \lVert\partial\mathcal L/\partial h_L\rVert)^{1/(L-1)}$.

Nó cũng tính một con số dự đoán, `pred`, mà mục 5 sẽ giải thích.

In [ ]:
def signal(f, init, depth, x=xb, y=yb, eps=EPS):
    """Mọi số đo của demo trên mạng chưa huấn luyện (giống compute() trong DeepSignalDemo.tsx)."""
    Ws, bs = init_net(init, depth, eps)
    hs, ds, z = forward(f, Ws, bs, x)
    gW, gb, gh = backward(Ws, hs, ds, z, y)
    n_in = [W.shape[0] for W in Ws[:-1]]
    sig_w = [sigma_of(init, n, Ws[0].shape[1]) for n in n_in]
    pred = np.array([s * np.sqrt(n) * np.sqrt((d**2).mean()) for s, n, d in zip(sig_w, n_in, ds)])  # mục 5
    if f == "relu":
        sat = np.array([(h <= 0).all(0).mean() for h in hs[1:]])  # nơ-ron chết: bằng 0 với cả lô
    else:
        lim = (0.03, 0.97) if f == "sigmoid" else (-0.97, 0.97)
        sat = np.array([((h < lim[0]) | (h > lim[1])).mean() for h in hs[1:]])
    return {
        "spread": np.array([np.sqrt(h.var(0).mean()) for h in hs[1:]]),  # ảnh còn khác nhau bao nhiêu
        "gh": gh,
        "pred": pred,
        "sat": sat,
        "loss": ce_loss(z, y),
        "gnorm": float(np.sqrt(sum((g**2).sum() for g in gW + gb))),
        "factor": (gh[0] / gh[-1]) ** (1 / (depth - 1)),
        "pred_factor": float(np.exp(np.log(pred[1:]).mean())),
        "ratio": gh[0] / gh[-1],
    }


FLOOR = 1e-15  # như demo: độ lệch chuẩn dưới mức này chỉ còn là sai số làm tròn, các ảnh đã như nhau


def report(f, init, depth):
    s = signal(f, init, depth)
    same = np.flatnonzero(s["spread"] < FLOOR)
    print(
        f"{f}/{init}, {depth} lớp: mỗi lớp gradient nhân với {num(s['factor'])} (dự đoán {num(s['pred_factor'])}); "
        f"lớp 1 so với lớp {depth}: {num(s['ratio'])}\n   khác nhau giữa ảnh: lớp 1 {num(s['spread'][0])}, "
        f"lớp {depth} {num(s['spread'][-1])}"
        + (f" (= 0 từ lớp {same[0] + 1})" if len(same) else "")
        + f"; {'nơ-ron chết' if f == 'relu' else 'bão hòa'} {pct(s['sat'].mean(), 0)}; mất mát {num(s['loss'])}"
    )
    return s

### 🤔 Dự đoán trước
Cấu hình mặc định của demo: sigmoid, khởi tạo Xavier, 20 lớp. Bài 02 nói mỗi $\sigma'$ không quá 0,25. Vậy gradient ở
lớp 1 so với lớp 20 nhỏ cỡ $0{,}25^{19} \approx 3{,}6 \cdot 10^{-12}$, lớn hơn số đó, hay nhỏ hơn? Và ở lượt xuôi,
50 ảnh khác nhau còn khác nhau đến lớp 20 không?

In [ ]:
s_sig = report("sigmoid", "xavier", 20)
print(f"0,25^19 = {sci(0.25**19)}")
# deep.py in ra cùng các tỉ số này khi tạo tệp của demo (live_stats, cùng 50 ảnh, cùng trọng số)
assert abs(s_sig["ratio"] / 8.53346e-13 - 1) < 1e-5, "khác với demo"
assert (
    f"{s_sig['factor']:.2f}" == "0.23" and f"{s_sig['pred_factor']:.2f}" == "0.24" and f"{s_sig['loss']:.2f}" == "2.50"
)
fwd = (s_sig["spread"][-1] / s_sig["spread"][0]) ** (1 / 19)
print(f"lượt xuôi: khác biệt giữa các ảnh co ×{vn(fwd, 3)} mỗi lớp (lượt ngược: ×{vn(s_sig['factor'], 3)})")
r10 = signal("sigmoid", "xavier", 10)["ratio"]
print(f"cùng cấu hình, 10 lớp: lớp 1 so với lớp 10 = {num(r10)}, tức nhỏ hơn {vn(1 / r10, 0)} lần")

Nhỏ hơn cả $0{,}25^{19}$: ở lớp 1, gradient chỉ còn $8{,}5 \cdot 10^{-13}$ lần gradient ở lớp 20. Đi ngược mỗi lớp nó bị
nhân với 0,232, thấp hơn 0,25 vì không phải nơ-ron nào cũng nằm đúng tại $z = 0$, nơi $\sigma'$ lớn nhất. Ngay cả với
10 lớp, gradient ở lớp đầu đã nhỏ hơn ở lớp cuối hàng trăm nghìn lần.

Lượt xuôi còn tệ hơn. Ở lớp 1, các ảnh khác nhau cỡ 0,081; ở lớp 20, chỉ còn $8{,}3 \cdot 10^{-14}$: cả 50 ảnh cho ra
gần như cùng một vectơ. Không có gì lạ: lượt xuôi và lượt ngược đi qua **cùng** các ma trận Jacobi, nên *sự khác biệt*
giữa hai đầu vào co lại theo gần đúng thừa số đó: ×0,234 mỗi lớp ở lượt xuôi, ×0,232 ở lượt ngược. Mạng đã quên
mất ảnh đầu vào trước khi kịp học. Và không có nơ-ron nào
bão hòa (0%): gradient biến mất ở đây không cần đến bão hòa.

Hình dưới vẽ cả hai lượt theo lớp, như hai biểu đồ của demo, với trục dọc thang log. Đường xanh (chấm tròn) là
sigmoid + Xavier; đường cam gạch (chấm vuông) là cấu hình demo ghim sẵn để so sánh, **ReLU + He**. Đường xám chấm
là “giữ nguyên độ lớn”: mức của lớp 1 ở lượt xuôi, mức của lớp 20 ở lượt ngược.

In [ ]:
s_relu = signal("relu", "he", 20)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
layers = np.arange(1, 21)
for ax, key, title in [
    (axes[0], "spread", "Lượt xuôi: 50 ảnh còn khác nhau bao nhiêu?"),
    (axes[1], "gh", "Lượt ngược: ‖∂L/∂hₗ‖"),
]:
    ax.plot(layers, s_sig[key], color=BLUE, marker="o", ms=4, lw=2, label="sigmoid · Xavier")
    ax.plot(layers, s_relu[key], color=ORANGE, marker="s", ms=4, lw=2, ls="--", label="ReLU · He")
    ref = s_sig[key][0] if key == "spread" else s_sig[key][-1]
    ax.axhline(ref, color=GREY, ls=":", lw=1.5, label="giữ nguyên độ lớn")
    ax.set_yscale("log")
    ax.set(title=title, xlabel="lớp l", xticks=[1, 5, 10, 15, 20])
    ax.grid(alpha=0.3)
axes[0].set_ylabel("độ lệch chuẩn giữa các ảnh")
axes[1].text(20, 1e-12, "← lượt ngược đi từ lớp 20 về lớp 1", ha="right", fontsize=9)
axes[0].legend(frameon=False, fontsize=9, loc="lower left")
plt.tight_layout()
vn_axes(fig)
plt.show()
print(f"ReLU · He: lớp 1 so với lớp 20 = {num(s_relu['ratio'])}; mỗi lớp nhân với {num(s_relu['factor'])}")
assert abs(s_relu["ratio"] / 0.889264 - 1) < 1e-5, "khác với demo"

Với ReLU + He, gradient đi qua 20 lớp mà gần như không đổi: lớp 1 nhận 0,889 lần gradient của lớp 20, tức mỗi lớp nhân
với 0,994. Đường sigmoid thì rơi thẳng xuống, ở cả hai biểu đồ. Phần tiếp theo hỏi: cái gì quyết định thừa số mỗi lớp,
và vì sao He lại cho đúng 1?

## 5 · Thừa số mỗi lớp: $\sigma\sqrt{n_{\text{in}}}\cdot\operatorname{rms}(f')$

Lượt ngược qua lớp $l$ (Bài 06) là $\partial\mathcal L/\partial h_{l-1} = \big(\partial\mathcal L/\partial h_l \odot f'(z_l)\big)\,W_l^\top$.
Giả sử các phần tử của $W_l$ độc lập, trung bình 0, độ lệch chuẩn $\sigma$, và $W_l$ có $n_{\text{in}}$ hàng. Mỗi phần tử
của vectơ kết quả là tổng của nhiều số hạng ngẫu nhiên, nên bình phương chuẩn của nó xấp xỉ

$$\Big\lVert \frac{\partial\mathcal L}{\partial h_{l-1}} \Big\rVert^2 \approx n_{\text{in}}\,\sigma^2\,\operatorname{rms}(f')^2\,\Big\lVert \frac{\partial\mathcal L}{\partial h_l} \Big\rVert^2
\quad\Longrightarrow\quad \text{thừa số mỗi lớp} \approx \sigma\sqrt{n_{\text{in}}}\cdot\operatorname{rms}(f'(z_l))$$

trong đó $\operatorname{rms}(f')$ là căn của trung bình $f'^2$ trên mọi nơ-ron và mọi ảnh. Đó chính là `pred` trong
`signal()`. Muốn thừa số bằng 1, ta chọn $\sigma$ theo hàm kích hoạt:
- **tanh** gần 0 có $f' \approx 1$, nên cần $\sigma = 1/\sqrt{n_{\text{in}}}$. Glorot & Bengio (2010) đòi cả lượt xuôi
  ($n_{\text{in}}\operatorname{Var}(w) = 1$) lẫn lượt ngược ($n_{\text{out}}\operatorname{Var}(w) = 1$), rồi chọn một
  điểm dung hòa: **Xavier**, $\operatorname{Var}(w) = 2/(n_{\text{in}} + n_{\text{out}})$. Ở lớp 64 → 64, ba điều kiện
  trùng nhau. (Bài báo lấy $w$ từ phân phối đều có cùng phương sai; demo dùng phân phối chuẩn.)
- **ReLU** có $f' = 1$ ở khoảng một nửa số nơ-ron và 0 ở nửa kia, nên $\operatorname{rms}(f') \approx \sqrt{1/2}$ và cần
  $\sigma = \sqrt{2/n_{\text{in}}}$: **He** (He và cộng sự, 2015), $\operatorname{Var}(w) = 2/n_{\text{in}}$.
- **sigmoid** có $f' \le 0{,}25$, nên cần $\sigma\sqrt{n_{\text{in}}} \ge 4$. Nhưng $\sigma$ lớn làm $z$ lớn, và $z$ lớn
  làm $f'$ nhỏ đi. Mục này đo xem hai lực đó gặp nhau ở đâu.

Ô dưới in $\sigma$ của bốn cách khởi tạo trong demo ở lớp 64 → 64.

In [ ]:
INITS = {"small": "nhỏ σ = 0,01", "xavier": "Xavier", "he": "He", "large": "lớn σ = 1"}
for init, label in INITS.items():
    s64 = sigma_of(init, 64, 64)
    print(f"{label:>12}: σ = {vn(s64, 3)},  σ·√64 = {vn(s64 * 8, 3)}")
assert f"{sigma_of('xavier', 64, 64):.3f}" == "0.125" and f"{sigma_of('he', 64, 64):.3f}" == "0.177"  # như demo

### 🤔 Dự đoán trước
Với tanh và $\sigma = 1$ (“lớn”), $z$ rất lớn nên phần lớn nơ-ron bão hòa: $|h| > 0{,}97$, ở đó $\tanh'(z) < 0{,}06$.
Gradient qua 20 lớp sẽ **biến mất** hay **bùng nổ**?

In [ ]:
s_tl = report("tanh", "large", 20)
rms_fp = s_tl["pred"][1:] / 8  # pred = σ·√n·rms(f′) với σ = 1, n = 64
print(f"rms(f′) ở các lớp 2–20: từ {vn(rms_fp.min(), 3)} đến {vn(rms_fp.max(), 3)};  σ·√n = 8")
assert f"{s_tl['factor']:.2f}" == "2.08" and f"{s_tl['sat'].mean():.2f}" == "0.78"

Bùng nổ: ×2,08 mỗi lớp, nên lớp 1 nhận gấp hơn một triệu lần gradient của lớp 20, dù 78% nơ-ron bão hòa.
“Bão hòa thì gradient bằng 0” là hiểu nhầm. $\operatorname{rms}(f')$ chỉ còn 0,248–0,279, vì vài nơ-ron không bão hòa
vẫn đóng góp $f' \approx 1$. Nhân với $\sigma\sqrt{n} = 8$, thừa số vẫn lớn hơn 2. Ngược lại, sigmoid + Xavier ở mục 4
có 0% nơ-ron bão hòa mà gradient vẫn biến mất. Bão hòa và gradient biến mất là hai chuyện khác nhau. Cái quyết định
là tích $\sigma\sqrt{n}\cdot\operatorname{rms}(f')$.

Ô dưới đo thừa số mỗi lớp cho cả 12 cấu hình của demo (3 hàm × 4 cách khởi tạo, 20 lớp), cạnh con số dự đoán.

In [ ]:
TABLE = {(f, init): signal(f, init, 20) for f in ["sigmoid", "tanh", "relu"] for init in INITS}
print("thừa số mỗi lớp, 20 lớp: đo (dự đoán σ√n·rms(f′))\n")
print(f"{'':>8}" + "".join(f"{label:>17}" for label in INITS.values()))
for f in ["sigmoid", "tanh", "relu"]:
    cells = [f"{num(TABLE[f, i]['factor'])} ({num(TABLE[f, i]['pred_factor'])})" for i in INITS]
    print(f"{f:>8}" + "".join(f"{c:>17}" for c in cells))
print(f"\nReLU · Xavier: lớp 1 so với lớp 20 = {num(TABLE['relu', 'xavier']['ratio'])}")
print(f"sigmoid · σ = 1: bão hòa {pct(TABLE['sigmoid', 'large']['sat'].mean(), 0)} số nơ-ron")
worst = max(abs(s["factor"] / s["pred_factor"] - 1) for s in TABLE.values())
print(f"đo và dự đoán lệch nhau nhiều nhất {pct(worst, 1)}")
# deep.py (live_stats) in ra các tỉ số này cho demo
assert abs(TABLE["tanh", "xavier"]["ratio"] / 0.309564 - 1) < 1e-5
assert abs(TABLE["relu", "xavier"]["ratio"] / 0.00119368 - 1) < 1e-5

Công thức một dòng dự đoán đúng thừa số đo được ở cả 12 cấu hình, sai không quá vài phần trăm. Ba hàng của bảng cho
ba bài học:
- **sigmoid** biến mất với mọi cách khởi tạo của demo, kể cả He. Chỉ $\sigma = 1$ ($\sigma\sqrt n = 8$) mới đưa
  thừa số lên 0,883, và cái giá là 50% số nơ-ron bão hòa.
- **tanh** cần Xavier (0,940) hoặc He (1,07): hai cách đều gần 1, một bên co nhẹ, một bên phình nhẹ.
- **ReLU** với Xavier co ×0,702 mỗi lớp, tức gradient ở lớp 1 chỉ còn 0,0012 lần lớp 20. Lý do là thiếu đúng một hệ
  số $\sqrt 2$: Xavier cho $\sigma\sqrt n = 1$, nhưng ReLU tắt khoảng một nửa nơ-ron nên $\operatorname{rms}(f') \approx 0{,}71$.
  He thêm đúng hệ số đó và cho 0,994.

Hình dưới quét liên tục cỡ khởi tạo $g = \sigma\sqrt{n_{\text{in}}}$ (dùng cho mọi lớp, kể cả lớp vào và lớp ra), rồi tìm
bằng phép chia đôi giá trị $g$ làm thừa số đúng bằng 1 cho từng hàm. Hai trục đều thang log. Vạch xám là $g$ của Xavier
và He ở lớp 64 → 64.

In [ ]:
gains = np.geomspace(0.25, 16, 25)
STYLE = {"sigmoid": (ORANGE, "s", "--"), "tanh": (BLUE, "o", "-"), "relu": (GREEN, "^", ":")}
fig, ax = plt.subplots(figsize=(8, 3.8))
crit = {}
for f, (color, marker, ls) in STYLE.items():
    label = "ReLU" if f == "relu" else f
    ax.plot(gains, [signal(f, g, 20)["factor"] for g in gains], color=color, marker=marker, ms=4, ls=ls, label=label)
    lo, hi = 0.25, 16.0  # chia đôi trên thang log: thừa số tăng theo g quanh điểm cắt 1
    for _ in range(40):
        mid = math.sqrt(lo * hi)
        lo, hi = (mid, hi) if signal(f, mid, 20)["factor"] < 1 else (lo, mid)
    crit[f] = lo
    s = signal(f, lo, 20)
    print(
        f"{f:>7}: thừa số = 1 khi g = {vn(lo, 3)};  lúc đó {'nơ-ron chết' if f == 'relu' else 'bão hòa'} "
        f"{pct(s['sat'].mean(), 0)}, rms(f′) = {vn(s['pred_factor'] / lo, 3)}"
    )
for g, name in [(1, "Xavier"), (math.sqrt(2), "He")]:
    ax.axvline(g, color=GREY, lw=1)
    ax.text(g * 1.03, 6, name, fontsize=9, color="#555555")
ax.axhline(1, color="black", lw=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(xlabel="cỡ khởi tạo g = σ·√n_in", ylabel="thừa số mỗi lớp", title="Thừa số mỗi lớp theo cỡ khởi tạo (20 lớp)")
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=0.3)
vn_axes(fig)
plt.show()
homog = signal("relu", 2.0, 20)["factor"] / signal("relu", 1.0, 20)["factor"]
print(f"ReLU: gấp đôi g thì thừa số gấp {vn(homog, 3)} lần;  sigmoid cần σ = g/√64 = {vn(crit['sigmoid'] / 8, 2)}")
assert abs(homog - 2) < 0.05 and abs(crit["relu"] - math.sqrt(2)) < 0.02 and 9 < crit["sigmoid"] < 11

- **ReLU** cắt đường 1 tại $g = 1{,}423$, sát $\sqrt 2 \approx 1{,}414$ của He. Đường của nó là một đường thẳng hoàn hảo
  trên thang log-log: gấp đôi $g$ thì thừa số gấp 2,019 lần. Vì $\operatorname{ReLU}(cz) = c\operatorname{ReLU}(z)$ với
  $c > 0$, co giãn mọi trọng số không đổi tập nơ-ron bật, nên mỗi lớp của lượt ngược nhân đúng thêm $c$; phần lệch
  khỏi 2 đến từ tín hiệu lỗi $p - \text{onehot}(y)$ ở đầu ra, vì softmax của logit lớn hơn đổi hướng của nó.
- **tanh** cắt tại $g = 1{,}124$, giữa Xavier và He. Vượt qua đó, đường cong thoải dần vì $z$ lớn bắt đầu làm tanh bão hòa.
- **sigmoid** cần $g = 10{,}07$, tức $\sigma \approx 1{,}26$ ở lớp 64 → 64, và khi đó hơn một nửa số nơ-ron đã bão hòa.
  Không có cỡ khởi tạo nào vừa giữ được gradient vừa giữ nơ-ron ở vùng hoạt động. Ngoài ra sigmoid luôn dương, nên
  mọi lớp sau nhận đầu vào có trung bình khác 0. Glorot & Bengio (2010) chỉ ra chính điểm này: sigmoid “không hợp với
  mạng sâu khởi tạo ngẫu nhiên”, và đẩy lớp ẩn trên cùng vào bão hòa.

### He giữ độ lớn, không giữ sự khác biệt

Ở hình mục 4, đường cam của ReLU + He nằm ngang ở lượt ngược, nhưng ở lượt xuôi nó vẫn đi xuống chậm. Hai đại lượng ở
đây khác nhau: **độ lớn** của tín hiệu (RMS của $h_l$) và **sự khác biệt giữa các ảnh** (độ lệch chuẩn giữa các ảnh,
biểu đồ trái). Ô dưới đo cả hai, cộng với độ giống nhau giữa hai ảnh: cosin trung bình giữa các vectơ $z_l$ của
50 ảnh (1 nghĩa là cùng hướng).

In [ ]:
def similarity(f, init, depth):
    Ws, bs = init_net(init, depth)
    hs, _, _ = forward(f, Ws, bs, xb)
    zs = [h @ W for h, W in zip(hs[:-1], Ws[:-1])]  # z_l = h_{l−1}·W_l (b = 0)
    unit = [z / np.linalg.norm(z, axis=1, keepdims=True) for z in zs]
    cos = [(u @ u.T)[np.triu_indices(len(u), 1)].mean() for u in unit]  # trung bình trên 1.225 cặp ảnh
    return np.sqrt([(h**2).mean() for h in hs[1:]]), np.sqrt([h.var(0).mean() for h in hs[1:]]), np.array(cos)


for f, init in [("relu", "he"), ("tanh", "xavier")]:
    rms, spread, cos = similarity(f, init, 20)
    print(
        f"{f}/{init}: độ lớn (RMS) {vn(rms[0], 3)} → {vn(rms[-1], 3)};  khác nhau giữa ảnh {vn(spread[0], 3)} → "
        f"{vn(spread[-1], 3)};  cosin giữa hai ảnh {vn(cos[0], 3)} → {vn(cos[-1], 3)}  (lớp 1 → lớp 20)"
    )

Với ReLU + He, **độ lớn** của tín hiệu gần như giữ nguyên qua 20 lớp (RMS từ 0,288 xuống 0,186, dao động quanh mức đó;
mạng hữu hạn nên không phẳng tuyệt đối). Nhưng hai ảnh bất kỳ ngày càng **giống nhau**: cosin giữa chúng tăng từ 0,317
lên 0,963. Mỗi ReLU cắt đi phần âm, nên các vectơ dồn dần về cùng một hướng. Với tanh + Xavier, cosin vẫn quanh 0,24:
tanh đối xứng quanh 0 và không có xu hướng đó. Khởi tạo đúng thang đo giữ được *độ lớn*, chưa chắc giữ được *thông tin*.
Schoenholz và cộng sự (2017) nghiên cứu chính hiện tượng này: trong một mạng ngẫu nhiên, thông tin chỉ truyền được qua
một độ sâu giới hạn, trừ khi mạng nằm sát ranh giới giữa “co” và “phình”.

## 6 · Huấn luyện thật: lớp đầu đứng yên, lớp cuối vẫn học

Đến giờ ta chỉ đo mạng *chưa huấn luyện*. Gradient nhỏ ở lớp đầu có thật sự làm hỏng việc học? Ô dưới huấn luyện đúng
như `reel/mnist_mlp/deep.py` làm cho demo: 1 epoch trên 55.000 ảnh, lô 64 ảnh theo cùng một thứ tự xáo trộn (seed 0),
SGD thường, có hoặc không **cắt gradient** (mục 7). Sau khi học, nó đo độ chính xác trên tập kiểm định và tập kiểm
tra, và mỗi lớp đã dịch đi bao xa so với lúc đầu: $\lVert W_l - W_l^{(0)}\rVert / \lVert W_l^{(0)}\rVert$.

In [ ]:
BATCH, CLIP = DEEP["batch"], DEEP["clip"]  # 64 ảnh mỗi lô; cắt chuẩn toàn cục của gradient ở 1
ORDER = np.random.default_rng(0).permutation(len(ytr))  # cùng thứ tự lô với deep.py


def accuracy(f, Ws, bs, X, y):
    return float((forward(f, Ws, bs, X)[2].argmax(1) == y).mean())


def rel_change(W, W0):
    """‖W − W₀‖ / ‖W₀‖, tính sao cho trọng số đã phình tới 10²⁰⁰ cũng không tràn số."""
    d = W - W0
    m, m0 = np.abs(d).max(), np.abs(W0).max()
    return float(m * np.linalg.norm(d / m) / (m0 * np.linalg.norm(W0 / m0))) if m > 0 else 0.0


def train(f, init, depth, eta, clip=False, eps=EPS):
    """1 epoch SGD như deep.py. Trả về độ chính xác, độ dịch của từng lớp và tham số; hoặc bước mà mất mát thành NaN."""
    Ws, bs = init_net(init, depth, eps)
    W0 = [W.copy() for W in Ws]
    with np.errstate(over="ignore", invalid="ignore"):  # cấu hình bùng nổ tràn số; ta phát hiện qua mất mát
        for step, i in enumerate(range(0, len(ORDER), BATCH), 1):
            idx = ORDER[i : i + BATCH]
            hs, ds, z = forward(f, Ws, bs, Xtr[idx])
            if not np.isfinite(ce_loss(z, ytr[idx])):
                return {"nan_step": step}
            gW, gb, _ = backward(Ws, hs, ds, z, ytr[idx])
            scale = 1.0
            if clip:  # cắt theo chuẩn toàn cục: chia mọi gradient cho cùng một số nếu ‖g‖ > 1
                norm = np.sqrt(sum((g**2).sum() for g in gW + gb))
                scale = min(1.0, CLIP / norm) if np.isfinite(norm) else 0.0
            for l in range(depth + 1):
                Ws[l] -= eta * scale * gW[l]
                bs[l] -= eta * scale * gb[l]
        if not all(np.isfinite(W).all() for W in Ws):
            return {"nan_step": step}
        return {
            "val": accuracy(f, Ws, bs, Xval, yval),
            "test": accuracy(f, Ws, bs, Xte, yte),
            "move": [rel_change(W, W_) for W, W_ in zip(Ws, W0)],
            "params": (Ws, bs),
        }


def demo_run(f, init, depth, clip=False):
    """Cấu hình tương ứng trong tệp của demo (η đã chọn trên tập kiểm định, như deep.py)."""
    return next(c for c in DEEP["configs"] if (c["act"], c["init"], c["depth"], c["clip"]) == (f, init, depth, clip))

### 🤔 Dự đoán trước
Huấn luyện mạng sigmoid + Xavier 20 lớp, với $\eta = 0{,}01$, mức mà demo chọn trên tập kiểm định. Sau một epoch
(860 bước), mạng đúng bao nhiêu phần trăm ảnh kiểm tra? Và trọng số lớp 1 dịch đi bao nhiêu so với độ lớn ban đầu?

In [ ]:
r_sig = train("sigmoid", "xavier", 20, eta=0.01)
pred_sig = forward("sigmoid", *r_sig["params"], Xte)[2].argmax(1)
print(f"{-(-len(ORDER) // BATCH)} bước; kiểm tra đúng {pct(r_sig['test'])}")
print(
    f"mạng đoán chữ số {np.unique(pred_sig).tolist()} cho cả {vn(len(pred_sig), 0)} ảnh; chữ số 1 chiếm {pct((yte == 1).mean())}"
)
print(
    f"lớp 1 dịch {num(r_sig['move'][0])} độ lớn ban đầu, lớp 20 dịch {num(r_sig['move'][19])}, lớp ra {num(r_sig['move'][-1])}"
)
d = demo_run("sigmoid", "xavier", 20)
print(f"demo: η = {vn(d['eta'], 2)}, kiểm tra {pct(d['test'])}, lớp 1 dịch {num(d['move'][0])}")
assert abs(r_sig["test"] - d["test"]) < 1e-9 and (pred_sig == 1).all(), "khác với demo"
assert abs(r_sig["move"][0] / d["move"][0] - 1) < 0.01 and abs(r_sig["move"][-1] / d["move"][-1] - 1) < 0.01
print(f"chữ số 1 trong tập huấn luyện: {vn((ytr == 1).sum(), 0)}/{vn(len(ytr), 0)} ảnh, nhiều nhất trong 10 chữ số")
assert np.bincount(ytr).argmax() == 1

11,35%, đúng bằng phần chữ số 1 trong tập kiểm tra: mạng trả lời “1” cho **mọi** ảnh. Trọng số lớp 1 dịch đi
$2{,}1 \cdot 10^{-13}$ lần độ lớn ban đầu, tức là đứng yên; lớp ra dịch 0,107, tức là vẫn học. Nhưng lớp ra học được
gì? Mục 4 cho thấy ở lớp 20, cả 50 ảnh đã cho ra gần như cùng một vectơ, nên thứ duy nhất lớp ra học được là
**chữ số nào hay gặp nhất**. Chữ số 1 là chữ số phổ biến nhất trong tập huấn luyện. “Lớp cuối vẫn học” đúng, nhưng
nó học trên một đầu vào đã mất hết thông tin về ảnh.

### He so với Xavier cho ReLU, ba seed

Giờ là phép thử của He và cộng sự (2015): cùng một mạng ReLU 20 lớp, khởi tạo He hay Xavier. Mỗi cách dùng $\eta$ mà
demo đã chọn trên tập kiểm định (He: 0,03; Xavier: 0,1). Để biết chênh lệch có vượt dao động ngẫu nhiên không, ta
chạy 3 seed của bộ sinh trọng số: seed 1 là mạng của demo, seed 2 và 3 cho các $\varepsilon$ khác. Thứ tự lô giữ nguyên.

In [ ]:
runs = {}
for init in ["he", "xavier"]:
    eta = demo_run("relu", init, 20)["eta"]
    for seed in [1, 2, 3]:
        eps = EPS if seed == DEEP["seed"] else draw_eps(seed)
        runs[init, seed] = train("relu", init, 20, eta, eps=eps)
    accs = [runs[init, s]["test"] for s in [1, 2, 3]]
    print(
        f"ReLU · {INITS[init]:<6} (η = {vn(eta, 2)}): {' · '.join(pct(a) for a in accs)}   khoảng {pct(min(accs))}–{pct(max(accs))}"
    )
he = [runs["he", s]["test"] for s in [1, 2, 3]]
xa = [runs["xavier", s]["test"] for s in [1, 2, 3]]
print(
    f"demo (seed 1): He {pct(demo_run('relu', 'he', 20)['test'])}, Xavier {pct(demo_run('relu', 'xavier', 20)['test'])}"
)
print("He so với Xavier:", "vượt dao động giữa các seed" if min(he) > max(xa) else "trong dao động giữa các seed")
print(
    f"thừa số mỗi lớp của ReLU · Xavier lúc đầu: {num(TABLE['relu', 'xavier']['factor'])}, mũ 19 = {num(TABLE['relu', 'xavier']['factor'] ** 19)}"
)
# Huấn luyện mạng sâu này nhạy với sai số làm tròn (mục 9), nên trên một CPU khác, seed 1 có thể lệch khỏi demo vài
# phần mười điểm: ta chỉ kiểm tra với dung sai rộng, cộng với kết luận chính.
assert abs(he[0] - demo_run("relu", "he", 20)["test"]) < 0.03 and min(he) > max(xa) + 0.1

He thắng Xavier vượt xa dao động giữa các seed: ba mạng He đạt 89,84–92,39%, ba mạng Xavier chỉ 11,35–50,18%, và một
mạng Xavier không học được gì (11,35%, lại là “luôn đoán 1”). Đó là phiên bản thu nhỏ của hình 3 trong bài báo của
He và cộng sự: với mạng 30 lớp, khởi tạo Xavier “hoàn toàn đứng yên”, còn khởi tạo của họ thì hội tụ. Chênh lệch
chỉ là một hệ số $\sqrt 2$ ở độ lệch chuẩn của các lớp 64 → 64 (0,177 so với 0,125), nhưng qua 19 lớp nó thành
$0{,}702^{19} = 0{,}0012$.

Hình dưới vẽ, cho ba lần chạy vừa rồi (seed 1, cũng là mạng của demo), mỗi lớp đã dịch đi bao xa:
$\lVert W_l - W_l^{(0)}\rVert / \lVert W_l^{(0)}\rVert$ theo lớp $l$, với lớp 21 là lớp ra. Trục dọc thang log.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for r, color, marker, ls, label in [
    (r_sig, BLUE, "o", "-", "sigmoid · Xavier"),
    (runs["xavier", 1], GREEN, "^", ":", "ReLU · Xavier"),
    (runs["he", 1], ORANGE, "s", "--", "ReLU · He"),
]:
    ax.plot(np.arange(1, 22), r["move"], color=color, marker=marker, ms=4, ls=ls, lw=1.8, label=label)
    print(f"{label:>16}: lớp 1 dịch {num(r['move'][0])}, lớp 20 dịch {num(r['move'][19])}, lớp ra {num(r['move'][20])}")
ax.set_yscale("log")
ax.set(xlabel="lớp (21 = lớp ra)", ylabel="‖W − W₀‖ / ‖W₀‖", title="Sau 1 epoch: mỗi lớp đã dịch đi bao xa?")
ax.set_xticks([1, 5, 10, 15, 20, 21])
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(alpha=0.3)
vn_axes(fig)
plt.show()

Với sigmoid, đường xanh leo từ $10^{-13}$ ở lớp 1 lên cỡ $10^{-1}$ ở lớp ra: mỗi lớp gần đầu ra hơn học nhiều hơn hẳn,
đúng hình ảnh “lớp đầu đứng yên, lớp cuối vẫn học”. Với ReLU, cả hai cách khởi tạo đều làm mọi lớp dịch đi từ vài phần
trăm đến vài chục phần trăm, kể cả lớp 1. Ở ReLU + Xavier, lớp đầu không đứng yên: tỉ số 0,0012 chỉ đúng lúc khởi tạo,
và nó học với $\eta = 0{,}1$ lớn hơn. Nhưng dịch nhiều không có nghĩa là học tốt: mạng này chỉ đạt 37,09%.

## 7 · Gradient bùng nổ và cắt gradient

Phía ngược lại: ReLU với $\sigma = 1$. Mỗi lớp nhân gradient với 5,67 (bảng mục 5).

### 🤔 Dự đoán trước
Mạng ReLU 20 lớp, $\sigma = 1$, chưa huấn luyện. Mất mát cross-entropy ban đầu trên 50 ảnh cỡ bao nhiêu? (Một mạng
đoán đều 10 chữ số có mất mát $\ln 10 \approx 2{,}30$.) Và huấn luyện nó với $\eta = 0{,}03$ thì sao?

In [ ]:
s_big = report("relu", "large", 20)
print(f"chuẩn của toàn bộ gradient: {num(s_big['gnorm'])};  ln 10 = {vn(math.log(10), 2)}")
r_big = train("relu", "large", 20, eta=0.03)
print(f"không cắt gradient, η = 0,03: mất mát thành NaN ở bước {r_big['nan_step']}")
d_big = demo_run("relu", "large", 20)
print("demo, mọi η:", " · ".join(f"η = {vn(s['eta'], 2)}: NaN ở bước {s['nanStep']}" for s in d_big["sweep"]))
assert r_big["nan_step"] == d_big["sweep"][1]["nanStep"] == 3, "khác với demo"

Mất mát ban đầu là $7{,}4 \cdot 10^{15}$ thay vì 2,30: các logit lớn khổng lồ, và mạng chắc chắn tuyệt đối vào câu trả
lời sai. Chuẩn của gradient là $2{,}0 \cdot 10^{16}$, nên bước SGD đầu tiên, dù $\eta$ nhỏ đến đâu trong lưới của demo,
đẩy trọng số đi cực xa. Đến bước 3 (bước 2 với $\eta = 1$), các số tràn khỏi phạm vi của số thực 64 bit và mất mát thành NaN
(“không phải số”).

**Cắt gradient** (gradient clipping) theo chuẩn toàn cục: nếu chuẩn $\lVert g\rVert$ của *toàn bộ* gradient (mọi $W$,
mọi $b$ ghép lại) vượt ngưỡng $c$, chia tất cả cho cùng một số để chuẩn còn đúng $c$:

$$g \leftarrow g \cdot \min\!\Big(1, \frac{c}{\lVert g\rVert}\Big)$$

Pascanu và cộng sự (2013) đề xuất cách này cho RNN, nơi gradient bùng nổ là chuyện thường ngày. Nó vẫn là tiêu chuẩn
khi huấn luyện mô hình ngôn ngữ lớn: GPT-3 và LLaMA đều “cắt chuẩn toàn cục của gradient ở 1,0”. Demo dùng $c = 1$.
Ô dưới huấn luyện lại cùng mạng, lần này có cắt.

In [ ]:
r_clip = train("relu", "large", 20, eta=0.03, clip=True)
d_clip = demo_run("relu", "large", 20, clip=True)
print(
    f"có cắt gradient, η = 0,03: kiểm tra đúng {pct(r_clip['test'])} (demo: {pct(d_clip['test'])}, η = {vn(d_clip['eta'], 2)})"
)
print(f"lớp 1 dịch {num(r_clip['move'][0])} độ lớn ban đầu, lớp ra dịch {num(r_clip['move'][-1])}")
W_norm = math.sqrt(sum((W**2).sum() for W in init_net("large", 20)[0]))
print(f"mỗi bước đi được tối đa η·c = {vn(0.03 * CLIP, 2)};  ‖W‖ ban đầu của cả mạng = {vn(W_norm, 0)}")
assert abs(r_clip["test"] - d_clip["test"]) < 0.01, "khác với demo"

Có cắt, mạng không còn nổ: mỗi bước đi được nhiều nhất $\eta c = 0{,}03$, rất nhỏ so với độ lớn 359 của trọng số. Nhưng
nó cũng gần như không học: 16,26%, và lớp 1 chỉ dịch 0,0021 độ lớn ban đầu. Cắt gradient chặn được những bước quá lớn,
chứ không sửa được một mạng mà mỗi lớp nhân tín hiệu với 5,67.

Nó cũng không sửa được gradient biến mất, vì hai lý do mà ô dưới in ra:
- cắt chia **mọi** gradient cho cùng một số, nên tỉ số giữa lớp 1 và lớp cuối không đổi;
- khi gradient biến mất, chuẩn toàn cục thường đã nhỏ hơn ngưỡng, nên phép cắt không bao giờ xảy ra.

In [ ]:
scale = CLIP / s_big["gnorm"]
print(
    f"ReLU · σ = 1: cắt nhân mọi gradient với {sci(scale)}; lớp 1 so với lớp 20 trước khi cắt {num(s_big['ratio'])}, "
    f"sau khi cắt {num((scale * s_big['gh'][0]) / (scale * s_big['gh'][-1]))}"
)
print(f"sigmoid · Xavier: ‖g‖ = {num(s_sig['gnorm'])} < {vn(CLIP, 0)}, nên không có gì để cắt")
d0, d1 = demo_run("sigmoid", "xavier", 20), demo_run("sigmoid", "xavier", 20, clip=True)
print(f"demo, sigmoid · Xavier · 20 lớp: không cắt {pct(d0['test'])}, có cắt {pct(d1['test'])}")
assert s_sig["gnorm"] < CLIP  # nhỏ hơn ngưỡng: phép cắt không bao giờ xảy ra

## 8 · Toàn bộ thí nghiệm của demo

Demo chạy đúng hàm `train()` ở trên cho mọi tổ hợp: 3 hàm kích hoạt × 4 cách khởi tạo × 4 độ sâu (5, 10, 20, 30 lớp)
× có/không cắt gradient × 5 giá trị $\eta$ (0,01 · 0,03 · 0,1 · 0,3 · 1), tức 480 lần huấn luyện, mất vài phút. Với mỗi
cấu hình, $\eta$ được chọn theo độ chính xác trên tập kiểm định, rồi đo một lần trên tập kiểm tra. Ta đọc kết quả từ
`DEEP["configs"]`. Một lần chạy phân kỳ được ghi là `{"eta": …, "nanStep": bước}` thay cho `"val"`; nếu mọi $\eta$
đều phân kỳ, `eta`, `val`, `test` và `move` là `null` (`None` trong Python). Tệp không chứa NaN hay vô cùng.

In [ ]:
DEPTHS, ACTS = DEEP["depths"], ["sigmoid", "tanh", "relu"]
n_runs = sum(len(c["sweep"]) for c in DEEP["configs"])
n_nan = sum("nanStep" in s for c in DEEP["configs"] for s in c["sweep"])
print(
    f"{len(DEEP['configs'])} cấu hình × {len(DEEP['etas'])} giá trị η = {n_runs} lần huấn luyện; {n_nan} lần mất mát thành NaN"
)


def cell(c):
    return "NaN" if c["test"] is None else vn(100 * c["test"], 1)


print(
    f"\nkiểm tra đúng (%), không cắt / có cắt; NaN = mọi η đều phân kỳ\n{'':>17}"
    + "".join(f"{d:>14} lớp" for d in DEPTHS)
)
for f in ACTS:
    for init, label in INITS.items():
        row = [f"{cell(demo_run(f, init, d))} / {cell(demo_run(f, init, d, True))}" for d in DEPTHS]
        print(f"{f:>7} {label:>12}" + "".join(f"{r:>18}" for r in row))

Hình dưới vẽ cùng bảng đó (không cắt gradient) theo độ sâu, mỗi hàm kích hoạt một ô. Mỗi cách khởi tạo có một màu,
một dạng điểm và một kiểu nét. Điểm “NaN” (mọi $\eta$ đều phân kỳ) được vẽ bằng dấu × ở đáy ô. Đường chấm xám ngang
là 11,35%, mức của “luôn đoán chữ số 1”.

In [ ]:
INIT_STYLE = {
    "small": (GREY, "x", (0, (1, 2))),
    "xavier": (BLUE, "o", "-"),
    "he": (ORANGE, "s", "--"),
    "large": (GREEN, "^", "-."),
}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), sharey=True)
for ax, f in zip(axes, ACTS):
    for init, (color, marker, ls) in INIT_STYLE.items():
        acc = [demo_run(f, init, d)["test"] for d in DEPTHS]
        ok = [a is not None for a in acc]
        ax.plot(
            [d for d, k in zip(DEPTHS, ok) if k],
            [100 * a for a, k in zip(acc, ok) if k],
            color=color, marker=marker, ls=ls, lw=1.8, ms=6, label=INITS[init],
        )  # fmt: skip
        if not all(ok):
            ax.scatter([d for d, k in zip(DEPTHS, ok) if not k], [2] * ok.count(False), color=color, marker="X", s=60)
    ax.axhline(100 * (yte == 1).mean(), color=GREY, ls=":", lw=1)
    ax.set(title="ReLU" if f == "relu" else f, xlabel="số lớp ẩn", xticks=DEPTHS, ylim=(0, 100))
    ax.grid(alpha=0.3)
axes[2].text(30, 6, "NaN", ha="center", fontsize=9)
axes[0].set_ylabel("kiểm tra đúng (%)")
axes[0].legend(frameon=False, fontsize=8, loc="center left")
plt.tight_layout()
vn_axes(fig)
plt.show()
for f, init in [("sigmoid", "large"), ("relu", "xavier"), ("relu", "he")]:
    print(f"{f} · {INITS[init]}: " + " · ".join(f"{d} lớp {cell(demo_run(f, init, d))}%" for d in DEPTHS))
small = [c["test"] for c in DEEP["configs"] if c["init"] == "small"]
print(f"σ = 0,01, cả {len(small)} ô: từ {pct(min(small), 1)} đến {pct(max(small), 1)}")
tanh_ok = [c["test"] for c in DEEP["configs"] if c["act"] == "tanh" and c["init"] in ("xavier", "he")]
print(f"tanh · Xavier hoặc He, cả {len(tanh_ok)} ô: thấp nhất {pct(min(tanh_ok), 1)}")
print(f"ReLU · σ = 1 · 10 lớp: có cắt {cell(demo_run('relu', 'large', 10, True))}%")

Đọc bảng và hình (mỗi ô là một seed, $\eta$ đã chọn trên tập kiểm định):
- **σ = 0,01** không học được gì ở bất kỳ độ sâu, hàm kích hoạt, hay $\eta$ nào: 11,3% ở cả 24 ô, tức “luôn đoán 1”
  (các đường của nó trùng nhau ở đáy mỗi ô, cùng đường sigmoid + Xavier).
  Mỗi lớp nhân gradient với cỡ 0,02–0,08, nên ngay cả 5 lớp đã đủ để lớp đầu đứng yên.
- **sigmoid** chỉ học được với σ = 1, và kém dần theo độ sâu: 86,2% ở 5 lớp, 27,0% ở 30 lớp: thừa số 0,883 mỗi lớp
  vẫn nhỏ hơn 1, và một nửa số nơ-ron đã bão hòa.
- **tanh** với Xavier hoặc He học tốt ở mọi độ sâu: thấp nhất 91,3%.
- **ReLU** với Xavier ổn ở 5–10 lớp (94,8% và 94,1%) rồi sụp ở 20–30 lớp (37,1% và 20,0%): mạng nông không “thấy” hệ số
  $\sqrt 2$, mạng sâu thì có. He và cộng sự cũng thấy đúng điều này: mạng 22 lớp hội tụ với cả hai cách, mạng 30 lớp
  thì không với Xavier. He giữ 92,4% ở 20 lớp, nhưng cũng tụt còn 80,4% ở 30 lớp (mục 10 bàn vì sao).
- **σ = 1 với ReLU** phân kỳ (NaN) với mọi $\eta$ từ 10 lớp trở lên. Cắt gradient đổi NaN thành một mạng chạy được
  (26,4% ở 10 lớp), nhưng không thành một mạng tốt.

Mỗi ô chỉ có một seed. Ở mục 6, ba seed của ReLU + He 20 lớp đã cách nhau vài điểm, nên chênh lệch nhỏ cỡ đó giữa có
và không cắt gradient có thể chỉ là dao động giữa các seed.

## 9 · Thử thay đổi

Các ô dưới chạy lại `signal()` và `train()` với tham số khác. Bạn có thể sửa giá trị rồi chạy lại. `signal()` chỉ là
một lượt xuôi và một lượt ngược trên 50 ảnh; mỗi lần `train()` là một epoch đầy đủ, nên chậm hơn nhiều.

### Độ rộng của mạng

Công thức $\sigma\sqrt{n}\cdot\operatorname{rms}(f')$ là một trung bình: nó đúng hơn khi mỗi lớp có nhiều nơ-ron. Ô dưới
đo thừa số mỗi lớp của ReLU + He, 20 lớp, với độ rộng 16, 64 và 256, mỗi độ rộng 3 seed của bộ sinh trọng số.

In [ ]:
by_width = {}
for width in [16, 64, 256]:
    fs = by_width[width] = [signal("relu", "he", 20, eps=draw_eps(seed, width, 20))["factor"] for seed in [1, 2, 3]]
    print(
        f"rộng {width:>3}: thừa số mỗi lớp {' · '.join(vn(v, 3) for v in fs)}  (khoảng {vn(min(fs), 3)}–{vn(max(fs), 3)})"
    )
f_low = min(by_width[16])
print(f"rộng 16, seed thấp nhất: {vn(f_low, 3)} mũ 19 = {num(f_low**19)}")

Trung bình đều quanh 1, nhưng mạng hẹp 16 nơ-ron dao động nhiều hơn hẳn giữa các seed; qua 19 lớp, một thừa số
0,892 đã thành 0,114. Đây là lý do “Xavier” và “He” là quy tắc về **phương sai**: chúng chỉ đúng trung bình, và càng
đúng khi mạng càng rộng.

### Độ sâu

Thừa số mỗi lớp gần như không đổi theo độ sâu; cái thay đổi là số lần nó được nhân. Ô dưới in tỉ số giữa lớp 1 và
lớp cuối ở 5, 10, 20 và 30 lớp.

In [ ]:
for f, init in [("sigmoid", "xavier"), ("relu", "xavier"), ("relu", "he"), ("tanh", "large")]:
    cells = [num(signal(f, init, d)["ratio"]) for d in DEPTHS]
    print(f"{f}/{init:<6}: " + " · ".join(f"{d} lớp {c}" for d, c in zip(DEPTHS, cells)))

Sigmoid + Xavier mất khoảng 6 bậc độ lớn sau mỗi 10 lớp; tanh với σ = 1 thêm khoảng 3 bậc. ReLU + He trôi chậm
(0,980 ở 10 lớp, 0,829 ở 30 lớp): thừa số 0,994 không phải đúng 1, và 29 lần nhân vẫn để lại dấu vết.

### Sai số làm tròn

Ô dưới huấn luyện lại ReLU + Xavier 20 lớp (seed 1, $\eta = 0{,}1$, như mục 6), nhưng nhân mọi $\varepsilon$ với
$1 + 10^{-14}\cdot$nhiễu: một thay đổi ở chữ số thập phân thứ 14, cỡ sai số làm tròn của phép tính số thực.

In [ ]:
rng = np.random.default_rng(7)
nudge_h = [e * (1 + 1e-14 * rng.standard_normal(e.shape)) for e in EPS[0]]  # ε₁ … ε₃₀
eps_nudged = (nudge_h, EPS[1] * (1 + 1e-14 * rng.standard_normal(EPS[1].shape)))  # rồi ε_ra
r_n = train("relu", "xavier", 20, demo_run("relu", "xavier", 20)["eta"], eps=eps_nudged)
print(f"ReLU · Xavier · 20 lớp: gốc {pct(runs['xavier', 1]['test'])}, trọng số lệch 10⁻¹⁴: {pct(r_n['test'])}")

Một thay đổi ở chữ số thứ 14 đổi kết quả sau một epoch, thường vài điểm phần trăm. Mạng sâu này nhạy cảm: những sai khác nhỏ
nhất bị khuếch đại qua 860 bước và 20 lớp. Vì thế khi bạn chạy notebook trên một máy khác (Colab dùng CPU x86, demo
được tạo trên aarch64), vài lần huấn luyện ở mục 6 có thể lệch khỏi số của demo một chút, dù cùng mã và cùng seed.
Kết luận thì không đổi: các ô `assert` chỉ kiểm tra chúng với dung sai rộng. Một chênh lệch muốn được tin phải lớn
hơn khoảng dao động giữa các seed, như He so với Xavier ở mục 6.

## 10 · Giới hạn và bước tiếp theo

Khởi tạo đúng thang đo chỉ lo cho **lúc bắt đầu**. Ô dưới đo lại thừa số mỗi lớp của mạng ReLU + He 20 lớp ở mục 6,
lần này *sau* 1 epoch huấn luyện, trên cùng 50 ảnh.

In [ ]:
Ws_he, bs_he = runs["he", 1]["params"]
hs, ds, z = forward("relu", Ws_he, bs_he, xb)
_, _, gh_after = backward(Ws_he, hs, ds, z, yb)
f_after = (gh_after[0] / gh_after[-1]) ** (1 / 19)
rms_after = [np.sqrt((h**2).mean()) for h in hs[1:]]
print(f"ReLU · He, 20 lớp: thừa số mỗi lớp lúc đầu {vn(s_relu['factor'], 3)}, sau 1 epoch {vn(f_after, 3)}")
print(f"độ lớn (RMS) của h: lớp 1 {vn(rms_after[0], 3)}, lớp 20 {vn(rms_after[-1], 3)} (lúc đầu 0,288 và 0,186)")

Sau một epoch, thừa số đã rời khỏi 1 và độ lớn của tín hiệu đã phình lên khoảng 4 lần từ lớp 1 đến lớp 20: huấn luyện
thay đổi trọng số, nên phân phối của $z$ ở mỗi lớp trôi đi, và không có gì kéo nó về lại. Ở 30 lớp, ReLU + He đã
tụt xuống 80,4% (mục 8). He và cộng sự cũng ghi nhận: mô hình 30 lớp của họ hội tụ, nhưng sai nhiều hơn mô hình
14 lớp (sai số top-1 trên ImageNet 38,56% so với 33,82%). Hai bài tới chữa đúng chỗ này:
- [Bài 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/chuan-hoa/): chuẩn hóa lại $z$
  ở **mỗi** lớp, trong suốt quá trình huấn luyện, thay vì chỉ chọn đúng thang đo một lần lúc đầu;
- [Bài 13 · Kết nối tắt & ResNet](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/resnet/): $h_l = h_{l-1} + F(h_{l-1})$,
  nên Jacobi của mỗi lớp là $I + \partial F/\partial h$ và gradient luôn có một đường đi thẳng về lớp đầu.

Gradient biến mất cũng không chỉ là chuyện của mạng nhiều lớp. Một RNN
([Bài 15](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rnn/)) dùng *cùng một* ma trận ở mọi bước
thời gian, nên qua 100 bước, gradient bị nhân 100 lần với cùng một thừa số. Đó chính là bài toán mà Hochreiter (1991)
và Bengio, Simard & Frasconi (1994) đã phân tích, trước khi có tên “học sâu”.

## Tóm tắt
- Gradient ở lớp đầu là tích của nhiều thừa số $w_l\,f'(z_l)$: 0,9 nhân 30 lần còn 0,042; 1,1 nhân 30 lần thành 17,45.
- Trên mạng 20 lớp của demo, sigmoid + Xavier nhân gradient với 0,232 mỗi lớp: lớp 1 chỉ nhận $8{,}5 \cdot 10^{-13}$ lần
  gradient của lớp 20, và 50 ảnh thành cùng một vectơ, dù không nơ-ron nào bão hòa.
- Thừa số mỗi lớp $\approx \sigma\sqrt{n_{\text{in}}}\cdot\operatorname{rms}(f')$. Xavier chọn $\operatorname{Var}(w) = 2/(n_{\text{in}} + n_{\text{out}})$
  cho tanh (0,940), He chọn $2/n_{\text{in}}$ cho ReLU (0,994); ReLU với Xavier thiếu đúng $\sqrt 2$ (0,702).
- Bão hòa không đồng nghĩa với gradient biến mất: tanh với σ = 1 có 78% nơ-ron bão hòa mà gradient vẫn bùng nổ ×2,08 mỗi lớp.
- Sau 1 epoch, ReLU 20 lớp đạt 89,84–92,39% với He nhưng 11,35–50,18% với Xavier (3 seed); sigmoid + Xavier đoán
  “1” cho mọi ảnh, với lớp 1 dịch $2{,}1 \cdot 10^{-13}$.
- Cắt gradient (chuẩn toàn cục ≤ 1, như GPT-3 và LLaMA) biến một mạng phân kỳ ở bước 3 thành một mạng chạy được,
  nhưng không sửa được gradient biến mất, cũng không sửa được một khởi tạo sai.

## Câu hỏi ôn tập
1. Trong chuỗi một nơ-ron mỗi lớp, thừa số mỗi lớp của gradient là gì? Vì sao trọng số lớn có thể bù lại giới hạn
   0,25 của $\sigma'$, và cái giá phải trả là gì?
2. Mạng sigmoid + Xavier 20 lớp có 0% nơ-ron bão hòa, vậy mà gradient vẫn biến mất. Còn tanh với σ = 1 có 78% nơ-ron
   bão hòa mà gradient lại bùng nổ. Giải thích cả hai bằng công thức $\sigma\sqrt{n_{\text{in}}}\cdot\operatorname{rms}(f')$.
3. Vì sao He dùng $\operatorname{Var}(w) = 2/n_{\text{in}}$ cho ReLU, gấp đôi $1/n_{\text{in}}$ của tanh? Nếu dùng Xavier cho
   ReLU, mỗi lớp mất bao nhiêu, và qua 19 lớp còn lại bao nhiêu?
4. Sau 1 epoch, mạng sigmoid + Xavier đúng 11,35% ảnh kiểm tra. Con số đó đến từ đâu? Lớp nào đã học, và học được gì?
5. Cắt gradient theo chuẩn toàn cục làm gì với tỉ số giữa gradient ở lớp 1 và ở lớp cuối? Vì sao nó chữa được gradient
   bùng nổ mà không chữa được gradient biến mất?
6. Với ReLU + He, độ lớn của tín hiệu gần như giữ nguyên qua 20 lớp. Vậy vì sao mạng 30 lớp vẫn kém mạng 20 lớp?

<details><summary>Gợi ý đáp án</summary>

1. $w_l\,f'(z_l)$, không chỉ $f'(z_l)$. Với $|w|$ lớn, tích có thể vượt 1 dù $f' \le 0{,}25$; nhưng $w$ lớn làm $z$ lớn,
   đẩy nơ-ron vào vùng bão hòa (σ = 1: 50% nơ-ron sigmoid bão hòa) và mạng khó học.
2. Sigmoid + Xavier: $\sigma\sqrt n = 1$ và $\operatorname{rms}(\sigma') < 0{,}25$, nên thừa số 0,232 dù không nơ-ron nào
   bão hòa. Tanh với σ = 1: $\sigma\sqrt n = 8$, còn $\operatorname{rms}(f') \approx 0{,}27$ nhờ vài nơ-ron chưa bão hòa;
   tích là 2,08 > 1.
3. ReLU tắt khoảng một nửa số nơ-ron, nên $\operatorname{rms}(f') \approx \sqrt{1/2}$; muốn tích bằng 1 cần
   $\sigma\sqrt{n} = \sqrt 2$. Với Xavier, mỗi lớp nhân 0,702, và qua 19 lớp còn 0,0012.
4. Đó là phần chữ số 1 trong tập kiểm tra: mạng đoán “1” cho mọi ảnh. Chỉ lớp ra học được (lớp 1 dịch
   $2{,}1 \cdot 10^{-13}$), nhưng đầu vào của nó đã giống nhau với mọi ảnh, nên nó chỉ học được chữ số nào hay gặp nhất.
5. Không đổi gì: mọi gradient bị chia cho cùng một số. Nó chặn các bước quá lớn, nên mạng không còn nổ thành NaN.
   Khi gradient biến mất, chuẩn toàn cục thường đã nhỏ hơn ngưỡng (sigmoid + Xavier: 0,830 < 1), nên không có gì để cắt.
6. He chỉ đúng lúc khởi tạo và chỉ giữ *độ lớn*: các ảnh vẫn dồn dần về cùng một hướng (cosin từ 0,317 lên 0,963 qua
   20 lớp), và sau khi huấn luyện, phân phối của $z$ trôi đi. Chuẩn hóa (Bài 10) và kết nối tắt (Bài 13) chữa phần này.
</details>

## Tìm hiểu thêm
- S. Hochreiter (1991), [*Untersuchungen zu dynamischen neuronalen Netzen*](https://people.idsia.ch/~juergen/SeppHochreiter1991ThesisAdvisorSchmidhuber.pdf),
  luận văn (tiếng Đức): phân tích đầu tiên về gradient biến mất.
- Y. Bengio, P. Simard & P. Frasconi (1994), [Learning long-term dependencies with gradient descent is difficult](https://doi.org/10.1109/72.279181).
- X. Glorot & Y. Bengio (2010), [Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html):
  sigmoid và bão hòa, khởi tạo Xavier (phương trình 12 và 16).
- K. He và cộng sự (2015), [Delving Deep into Rectifiers](https://arxiv.org/abs/1502.01852): khởi tạo He; hình 2–3 so
  sánh với Xavier trên mạng 22 và 30 lớp.
- R. Pascanu, T. Mikolov & Y. Bengio (2013), [On the difficulty of training recurrent neural networks](https://arxiv.org/abs/1211.5063):
  cắt gradient theo chuẩn.
- S. Schoenholz và cộng sự (2017), [Deep Information Propagation](https://arxiv.org/abs/1611.01232): thông tin truyền qua
  mạng ngẫu nhiên được bao xa.
- T. Brown và cộng sự (2020), [Language Models are Few-Shot Learners](https://arxiv.org/abs/2005.14165) (GPT-3, phụ lục B) và
  H. Touvron và cộng sự (2023), [LLaMA](https://arxiv.org/abs/2302.13971) (mục 2.3): cắt chuẩn gradient ở 1,0.
- deeplearning.ai, [Initializing neural networks](https://www.deeplearning.ai/ai-notes/initialization/) (tương tác);
  [Dive into Deep Learning, mục 5.4](https://d2l.ai/chapter_multilayer-perceptrons/numerical-stability-and-init.html);
  ghi chú CS231n, [Neural Networks Part 2](https://cs231n.github.io/neural-networks-2/) (khởi tạo).

**Bài tiếp theo:** [Bài 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/chuan-hoa/):
chuẩn hóa $z$ ở mọi lớp trong suốt quá trình huấn luyện, để tín hiệu giữ được thang đo không chỉ lúc khởi tạo.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")